In [1]:
# 07_robustness.ipynb
# =============================================================================
# Consolidation and robustness. Pulls the per-experiment result tables into one
# master summary, checks robustness of every headline number to the leakage
# mask (strict A vs as-shipped B), reports between-seed stability, and adds a
# STRATIFIED analysis: does the contagion uplift over the intra-only model
# concentrate in firms whose own lawsuit record is thin (where relational
# structure must carry the signal) rather than in lawsuit-rich firms? This
# directly tests the paper's central reading that predictive value lies in
# relational structure, not in per-lawsuit detail. Figures grayscale, dpi 600,
# png + pdf, no caption.
# =============================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from torch_geometric.nn import HeteroConv, SAGEConv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[0]
PROCESSED = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "savefig.bbox": "tight",
                     "font.size": 10})
sns.set_theme(style="whitegrid", palette="Greys_r")
SEEDS = list(range(5))


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)


# --- 1. Master results table from per-notebook CSVs --------------------------
def safe(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

base = safe(TAB / "02_baselines.csv")
arch = safe(TAB / "03_arch_comparison.csv")
dec = safe(TAB / "04_decomposition.csv")
enr = safe(TAB / "05_enrichment.csv")

master = []
for _, r in base[base["mask"] == "A"].iterrows():
    master.append({"RQ": "RQ1", "config": f"{r.tier}:{r.model}({r.features})",
                   "AUC_mean": r.AUC_mean, "AUC_std": r.AUC_std})
for _, r in arch[arch["mask"] == "A"].iterrows():
    master.append({"RQ": "RQ1", "config": f"GNN:{r.arch}", "AUC_mean": r.AUC_mean,
                   "AUC_std": r.AUC_std})
for _, r in dec.iterrows():
    master.append({"RQ": "RQ2", "config": r.config, "AUC_mean": r.AUC_mean,
                   "AUC_std": r.AUC_std})
for _, r in enr[enr["mask"] == "A"].iterrows():
    master.append({"RQ": "RQ3", "config": r.condition, "AUC_mean": r.AUC_mean,
                   "AUC_std": r.AUC_std})
master_df = pd.DataFrame(master).round(4)
master_df.to_csv(TAB / "07_master_results.csv", index=False)
print("=== master results (mask A, test AUC) ===")
print(master_df.to_string(index=False))

# --- 2. Mask A vs B robustness (key models) ----------------------------------
ab = []
for df, key, col in [(arch, "arch", "arch"), (enr, "enrichment", "condition")]:
    for _, r in df.iterrows():
        ab.append({"family": key, "name": r[col], "mask": r["mask"], "AUC": r.AUC_mean})
ab_df = pd.DataFrame(ab)
piv = ab_df.pivot_table(index=["family", "name"], columns="mask", values="AUC").reset_index()
piv["A_minus_B"] = (piv["A"] - piv["B"]).round(4)
piv.to_csv(TAB / "07_mask_robustness.csv", index=False)
print("\n=== leakage-mask robustness (A strict vs B as-shipped) ===")
print(piv.to_string(index=False))
print(f"mean |A-B| gap = {piv['A_minus_B'].abs().mean():.4f} "
      f"(small gap -> conclusions robust to the leakage handling)")

fig, ax = plt.subplots(figsize=(5.4, 3.4))
lab = [f"{n}" for n in piv["name"]]
xp = np.arange(len(piv)); width = 0.38
ax.bar(xp - width / 2, piv["A"], width, color="#333333", edgecolor="black",
       linewidth=0.5, label="Mask A (strict)")
ax.bar(xp + width / 2, piv["B"], width, color="#999999", edgecolor="black",
       linewidth=0.5, label="Mask B (as-shipped)")
ax.set_xticks(xp); ax.set_xticklabels(lab, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Test AUC"); ax.set_ylim(0.6, 0.82)
ax.legend(frameon=True, fontsize=8)
savefig(fig, "fig_mask_robustness")

# --- 3. Stratified contagion vs intra by lawsuit count -----------------------
data = torch.load(PROCESSED / "smesd_hetero.pt", weights_only=False)
comp = data["company"]
raw_x = comp.x.numpy().astype(np.float64)                 # raw (unscaled) features
lawsuit_count = raw_x[:, 3:14].sum(1)                      # 11 cause counts
Xc = raw_x.copy(); Xc[:, :2] = np.log1p(np.clip(Xc[:, :2], 0, None))
Xc = StandardScaler().fit(Xc[comp.train_mask.numpy()]).transform(Xc).astype(np.float32)
comp.x = torch.tensor(Xc)
y = comp.y
trm, vam, tem = comp.train_mask, comp.val_mask, comp.test_mask
EDGE_TYPES = [et for et in data.edge_types if int(et[1].split("_")[1]) <= 11]
ei_dict = {et: data[et].edge_index for et in EDGE_TYPES}
IN_DIMS = {"company": comp.x.size(1), "person": data["person"].x.size(1)}


class HeteroGNN(nn.Module):
    def __init__(self, hidden=64, out=2, layers=2, dropout=0.3):
        super().__init__()
        self.drop = dropout
        self.lin_in = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in IN_DIMS.items()})
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((hidden, hidden), hidden) for et in EDGE_TYPES},
                       aggr="sum") for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, out))

    def forward(self, xd, eid):
        h = {nt: F.relu(self.lin_in[nt](x)) for nt, x in xd.items()}
        for conv in self.convs:
            h = conv(h, eid)
            h = {nt: F.dropout(F.relu(v), p=self.drop, training=self.training)
                 for nt, v in h.items()}
        return self.head(h["company"])


class IntraMLP(nn.Module):
    def __init__(self, d, hidden=64, out=2, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, out))

    def forward(self, x):
        return self.net(x)


def class_w():
    cls = torch.bincount(y[trm], minlength=2).float()
    return cls.sum() / (2 * cls.clamp(min=1))


def probs(kind, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    w = class_w()
    if kind == "contagion":
        model = HeteroGNN(); xd = {nt: data[nt].x for nt in data.node_types}
        fwd = lambda: model(xd, ei_dict)
    else:
        model = IntraMLP(comp.x.size(1)); fwd = lambda: model(comp.x)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-4)
    best_val, best_state, wait = -1, None, 0
    for _ in range(200):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(fwd()[trm], y[trm], weight=w); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            p = F.softmax(fwd(), 1)[:, 1].numpy()
        va = roc_auc_score(y[vam].numpy(), p[vam.numpy()])
        if va > best_val:
            best_val, best_state, wait = va, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 30:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        return F.softmax(fwd(), 1)[:, 1].numpy()


yt = y[tem.numpy()].numpy()
lc_test = lawsuit_count[tem.numpy()]
p_intra = np.mean([probs("intra", s) for s in SEEDS], axis=0)[tem.numpy()]
p_cont = np.mean([probs("contagion", s) for s in SEEDS], axis=0)[tem.numpy()]

# strata: no lawsuits / few (1) / many (>=2)
strata = {"0 lawsuits": lc_test == 0, "1 lawsuit": lc_test == 1, ">=2 lawsuits": lc_test >= 2}
strat_rows = []
for nm, m in strata.items():
    if m.sum() >= 10 and len(np.unique(yt[m])) == 2:
        strat_rows.append({"stratum": nm, "n": int(m.sum()),
                           "AUC_intra": round(roc_auc_score(yt[m], p_intra[m]), 4),
                           "AUC_contagion": round(roc_auc_score(yt[m], p_cont[m]), 4)})
strat_df = pd.DataFrame(strat_rows)
strat_df["uplift"] = (strat_df["AUC_contagion"] - strat_df["AUC_intra"]).round(4)
strat_df.to_csv(TAB / "07_stratified_lawsuit.csv", index=False)
print("\n=== stratified contagion vs intra by own lawsuit count (mask A, test) ===")
print(strat_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(5.2, 3.4))
xp = np.arange(len(strat_df)); width = 0.38
ax.bar(xp - width / 2, strat_df["AUC_intra"], width, color="#999999", edgecolor="black",
       linewidth=0.5, label="Intra-only")
ax.bar(xp + width / 2, strat_df["AUC_contagion"], width, color="#333333", edgecolor="black",
       linewidth=0.5, label="Contagion")
ax.set_xticks(xp); ax.set_xticklabels([f"{r.stratum}\n(n={r.n})" for _, r in strat_df.iterrows()],
                                      fontsize=8)
ax.set_ylabel("Test AUC (mask A)"); ax.set_ylim(0.4, 0.9)
ax.legend(frameon=True, fontsize=8)
savefig(fig, "fig_stratified")

print("\n07_robustness complete: master table, mask A/B robustness, stratified "
      "analysis; 2 figures, 3 tables.")


=== master results (mask A, test AUC) ===
 RQ                  config  AUC_mean  AUC_std
RQ1    Floor:LogReg(basic3)    0.6235   0.0000
RQ1       Floor:MLP(basic3)    0.6564   0.0508
RQ1   Intra:LogReg(intra23)    0.6216   0.0000
RQ1      Intra:MLP(intra23)    0.6813   0.0087
RQ1 Intra:LightGBM(intra23)    0.6747   0.0037
RQ1  Intra:XGBoost(intra23)    0.6807   0.0043
RQ1                GNN:ATTN    0.7455   0.0186
RQ1                GNN:SAGE    0.7658   0.0135
RQ2                   intra    0.6862   0.0083
RQ2          contagion_0_11    0.7671   0.0136
RQ2          contagion_0_19    0.7658   0.0135
RQ3                 C0_base    0.7658   0.0135
RQ3                  C1_sem    0.7670   0.0154
RQ3                  C2_dir    0.7575   0.0135
RQ3                 C3_both    0.7656   0.0119

=== leakage-mask robustness (A strict vs B as-shipped) ===
    family    name      A      B  A_minus_B
      arch    ATTN 0.7455 0.7576    -0.0121
      arch    SAGE 0.7658 0.7813    -0.0155
enrichment C0_


=== stratified contagion vs intra by own lawsuit count (mask A, test) ===
     stratum   n  AUC_intra  AUC_contagion  uplift
  0 lawsuits 272     0.7163         0.7759  0.0596
   1 lawsuit  38     0.6277         0.8092  0.1815
>=2 lawsuits 181     0.6957         0.7976  0.1019

07_robustness complete: master table, mask A/B robustness, stratified analysis; 2 figures, 3 tables.
